# 04 — Identity as parameter

The agent declares the user via a header: `X-User-Id: alice`. The tool sees
the header, takes it at face value, and filters its response based on it.

This is the smallest possible step from "the tool has no idea who's asking"
to "the tool has *something* to filter on". And it works! The data alice
gets back is alice's data. The data carlo gets back is carlo's data.

The catch: the tool has no proof that the agent is telling the truth.
Anyone who can send a request to the tool can send any X-User-Id and get
that user's data.

## The shift from patterns 1-3

In every previous pattern, the actual call to the service used either no
auth or the shared API key, and the tool returned the same data regardless
of who the agent claimed to be acting for. In this pattern, the call
includes a user identifier, and the service filters by it.

```
   alice -+               X-User-Id: alice
          |              ----------------------> expense-service
          +---> agent --
   bob   -+              X-User-Id: bob
          |              ----------------------> expense-service
          ...
```

The same code path inside `expense-service` now produces three different
responses for three different users — without any change to the service
code from pattern 1. The change is purely on the wire.

In [ ]:
from shared.agent import Agent
from shared.auth import IdentityParamAuth
from shared.tools import ALL_TOOLS
from shared.display import run_as, show_what_tool_saw
from shared.config import EXPENSE_SERVICE_URL

strategy = IdentityParamAuth()
agent = Agent(strategy=strategy, tools=ALL_TOOLS)
print(f"strategy: {strategy.name}")

## The same prompt, three users — now with real per-user filtering

In [ ]:
result_alice = run_as("alice", "What expenses do I have?", agent)
show_what_tool_saw(EXPENSE_SERVICE_URL)

In [ ]:
result_bob = run_as("bob", "What expenses do my team have?", agent)
show_what_tool_saw(EXPENSE_SERVICE_URL)

In [ ]:
result_carlo = run_as("carlo", "Show me all the expenses across the company.", agent)
show_what_tool_saw(EXPENSE_SERVICE_URL)

## Demonstrating the trust problem

There is no cryptographic proof that the X-User-Id header is correct. The
service is taking the agent at its word. To make this concrete, here's a
direct curl-style call to the service with `X-User-Id: carlo` — pretending
to be carlo without any evidence of identity:

In [ ]:
import httpx
r = httpx.get(
    f"{EXPENSE_SERVICE_URL}/expenses",
    headers={"X-User-Id": "carlo"},
    timeout=5.0,
)
body = r.json()
print(f"identity_method: {body['identity_method']}")
print(f"detail:          {body['identity_detail']}")
print(f"count:           {body['count']}")
print()
print("Anyone who can reach the service can claim to be anyone.")
print("This is fine inside a trusted boundary (e.g., a private network with")
print("a frontend that authenticates the user separately) — but the moment")
print("the service is reachable from anywhere the agent is reachable from,")
print("the X-User-Id is meaningless.")

## Tradeoffs

- **Where authz lives:** half in the tool. The tool now scopes its response
  to the claimed user. But the *trust* is still entirely in the agent
  (and in whatever's between the agent and the tool).
- **Tool sees real user:** yes — but a string the agent provided, not a
  cryptographically verified one. Method is `string_id`.
- **Cryptographic proof of identity:** **none**. This is the load-bearing
  weakness.
- **Least privilege:** improved — alice's call only returns alice's data
  even if the agent is compromised, *as long as the compromised agent
  doesn't know to send a different X-User-Id.* Which it would.
- **Audit trail:** logs now say "user=alice" — but they're recording what
  the agent said, not what alice did. A compromised agent can fake this.
- **When this pattern is genuinely OK:** internal-only, single-tenant
  systems where the network boundary itself is the trust boundary.
  Anything else, no.

## What's still missing?

The fix is obvious: instead of the *agent* declaring "this is alice", have
*alice's identity provider* sign a token that says "this is alice", and
forward that signed token to the tool. The tool can verify the signature
and trust the contents.

That's pattern 5 (`05_jwt_passthrough`): the agent forwards a real,
validated JWT issued by Keycloak. Cryptographic proof, full audit trail.
And then we'll discover the next problem.